## Power Iteration - CuPy - Asynchrony - SOLUTION

### Table of Contents
1. [Introduction and Setup](#1-Introduction-and-Setup)
2. [Theory: Streams and Synchronization](#2-Theory:-Streams-and-Synchronization)
3. [The Baseline Implementation](#3-The-Baseline-Implementation)
4. [Profiling the Baseline](#4-Profiling-the-Baseline)
5. [Better Visibility with NVTX](#5-Better-Visibility-with-NVTX)
6. [Implementing Asynchrony](#6-Implementing-Asynchrony)
7. [Performance Analysis](#7-Performance-Analysis)
8. [Balancing CPU I/O and GPU Compute](#8-Balancing-CPU-I/O-and-GPU-Compute)

### 1. Introduction and Setup

GPU programming is inherently asynchronous. In this exercise, we will explore the implications of this behavior when using CuPy and learn how to analyze the flow of execution using profiling tools.

We will revisit the Power Iteration algorithm. Our goal is to take a standard implementation, profile it to identify bottlenecks caused by implicit synchronization, and then optimize it using CUDA streams and asynchronous memory transfers.

First, we need to ensure the Nsight Systems profiler (nsys), Nsightful, and NVTX are installed and available.

In [ ]:
import os

# Install necessary packages if running in Google Colab.
if os.getenv("COLAB_RELEASE_TAG") and not os.path.exists("/accelerated-computing-hub-installed"):
  print("Downloading Nsight Systems package.")
  !curl -s -L -O https://developer.nvidia.com/downloads/assets/tools/secure/nsight-systems/2026_1/NsightSystems-linux-cli-public-2026.1.1.204-3717666.deb
  print("Installing Nsight Systems package.")
  !dpkg -i NsightSystems-linux-cli-public-2026.1.1.204-3717666.deb > /dev/null
  print("Installing PIP packages.")
  !pip install "nvtx" "nsightful[notebook] @ git+https://github.com/robobryce/nsightful.git@2e59cafafc9c3ba7a11740d82f79cfa11d47ee3c" > /dev/null 2>&1
  open("/accelerated-computing-hub-installed", "a").close()
  print("All packages installed.")

import numpy as np
import cupy as cp
import cupyx as cpx
import nvtx
from dataclasses import dataclass
from jupyter_dark_detect import is_dark
import matplotlib.pyplot as plt
plt.style.use('dark_background' if is_dark() else 'default')

### 2. Theory: Streams and Synchronization

All GPU work is launched asynchronously on a stream. The work items in a stream are executed in order. If you launch `f` on a stream and later launch `g` on that same stream, then `f` will be executed before `g`. But if `f` and `g` are launched on different streams, then their execution might overlap.

**How CuPy handles this:**

- **Default Stream:** Unless specified, CuPy launches work on the default CUDA stream.
- **Sequential Device Execution:** By default, CuPy work executes sequentially on the GPU.
- **Asynchronous Host Execution:** From the Python (Host) perspective, the code often returns immediately after launching the GPU kernel, before the work is actually finished.

**SOLUTION:** Certain operations force the CPU to wait for the GPU to finish (implicit synchronization):
- Accessing element values from device arrays (e.g., `x[0]`)
- Printing device array values
- Device-to-host memory transfers with `cp.asnumpy()` (by default)
- Explicit synchronization calls

### 3. The Baseline Implementation

We will start with a baseline implementation of the Power Iteration algorithm.

The setup below mirrors the previous memory-spaces notebook: one configuration, one generated matrix, and one estimator function. The Nsight Systems kernel lets us profile these ordinary notebook cells directly.

In [ ]:
@dataclass
class PowerIterationConfig:
  dim: int = 19000
  dominance: float = 0.05
  max_steps: int = 400
  check_frequency: int = 25
  progress: bool = True
  residual_threshold: float = 1e-10

In [ ]:
def generate_device(cfg=PowerIterationConfig()):
  cp.random.seed(42)
  weak_lam = cp.random.random(cfg.dim - 1) * (1.0 - cfg.dominance)
  lam = cp.random.permutation(cp.concatenate((cp.asarray([1.0]), weak_lam)))
  P = cp.random.random((cfg.dim, cfg.dim))
  D = cp.diag(cp.random.permutation(lam))
  return (P @ D) @ cp.linalg.inv(P)

In [ ]:
A_device = generate_device()

In [ ]:
def estimate_device_baseline(A, cfg=PowerIterationConfig()) -> np.ndarray:
  with nvtx.annotate("Setup"):
    A_gpu = cp.asarray(A)
    x = cp.ones(A_gpu.shape[0], dtype=np.float64)

  with nvtx.annotate("Loop"):
    for i in range(0, cfg.max_steps, cfg.check_frequency):
      with nvtx.annotate(f"Step {i} to {i + cfg.check_frequency}"):
        with nvtx.annotate(f"Compute & Residual {i}"):
          y = A_gpu @ x
          lam = (x @ y) / (x @ x)
          res = cp.linalg.norm(y - lam * x)
          x = y / cp.linalg.norm(y)

        with nvtx.annotate(f"Copy {i}"):
          x_host = cp.asnumpy(x)

        with nvtx.annotate(f"I/O {i}", payload=i):
          if cfg.progress:
            print(f"step {i}: residual = {res:.3e}")
          np.savetxt(f"/tmp/device_{i}.txt", x_host)

        if res < cfg.residual_threshold:
          break

        with nvtx.annotate(f"Compute {i + 1} to {i + cfg.check_frequency}"):
          for j in range(i + 1, min(i + cfg.check_frequency, cfg.max_steps)):
            with nvtx.annotate(f"Compute Step {j}"):
              y = A_gpu @ x
              x = y / cp.linalg.norm(y)

  return cp.asnumpy((x.T @ (A_gpu @ x)) / (x.T @ x))

estimate_device_baseline(
  A_device,
  cfg=PowerIterationConfig(max_steps=1, check_frequency=1, progress=False),
)

### 4. Profiling the Baseline

This notebook uses the **Python 3 (Nsight Systems)** kernel so we can profile the baseline in place. The `%%nsys` magic collects only this cell and displays the resulting timeline without restarting the kernel.

In [ ]:
%%nsys -o power_iteration__baseline.nsys-rep
lam_est_baseline = estimate_device_baseline(A_device)
np.testing.assert_allclose(lam_est_baseline, 1, atol=1e-4)

The timeline is displayed below the profiled cell. Explore what's going on in the program.

**EXTRA CREDIT:** Download the Nsight Systems GUI and open the report in it to see even more information.

In [ ]:
# The native report is saved as power_iteration__baseline.nsys-rep.

### 5. Better Visibility with NVTX

Nsight Systems shows us a lot of information—sometimes too much, and not all of it is relevant. We can annotate specific regions of our code so they stand out in the timeline. These regions can have categories, domains, and colors, and they can be nested. To add them, use the `nvtx.annotate()` context manager:

```
with nvtx.annotate("Loop"):
  for i in range(20):
    with nvtx.annotate(f"Step {i}"):
      pass
```

**SOLUTION:** The baseline code above includes `nvtx.annotate()` regions for the setup, loop, step, compute and residual, copy, I/O, and compute phases.

From our profile trace, we can see that both our CPU and GPU are idly waiting for each other! Device code is idle during every I/O step when we print the residual and write the checkpoint, and host code spends a long time synchronizing on `cudaMemcpyAsync`.

Here's what happens at the start of each I/O step:

- We copy from device to host, which synchronizes with any outstanding work on the device. This blocks the host for awhile.
- After that synchronous transfer has completed, we begin the I/O (printing and writing the checkpoint). During this time, the device is idle.
- After the I/O has completed on the host, we start launching the next set of iterations.

This is inefficient; we can do better by overlapping compute and I/O:

- First, host code asynchronously initiates our device-to-host copies.
- Then, host code asynchronously launches the next set of compute steps on the device.
- Next, host code synchronizes with the asynchronous copies we started.
- Finally, the host performs the I/O while the device performs the next set of compute steps.

Everything is still going to run on one stream, but we want to be able to synchronize with just the I/O, which is launched on the stream before the compute work. We'll use a CUDA event, which we will record on the stream right after the copy. Then, we can synchronize with the event later, waiting for the I/O but not the compute!

### 6. Implementing Asynchrony

Remember what we've learned about streams and how to use them with CuPy:

- By default, all CuPy operations within a single thread run on the same stream. You can access this stream with `cp.cuda.get_current_stream()`.
- You can create a new stream with `cp.cuda.Stream(non_blocking=True)`. Use `with` statements to use the stream for all CuPy operations within a block.
- You can record an event on a stream by calling `.record()` on it.
- You can synchronize on an event (or an entire stream) by calling `.synchronize()` on it.
- Memory transfers will block by default. You can launch them asynchronously with `cp.asarray(..., blocking=False)` (for host to device transfers) and `cp.asnumpy(..., blocking=False)` (for device to host transfers).

**SOLUTION:** The implementation below uses asynchronous memory transfers and CUDA events to overlap compute and I/O operations.

In [ ]:
def estimate_device_async(A, cfg=PowerIterationConfig()) -> np.ndarray:
  with nvtx.annotate("Setup"):
    A_gpu = cp.asarray(A) # If `A` is on the host, copy from host to device.
                          # Otherwise, does nothing.

    x = cp.ones(A_gpu.shape[0], dtype=np.float64)

  with nvtx.annotate("Loop"):
    for i in range(0, cfg.max_steps, cfg.check_frequency):
      with nvtx.annotate(f"Step {i} to {i + cfg.check_frequency}"):
        with nvtx.annotate(f"Compute & Residual {i}"):
          y = A_gpu @ x
          lam = (x @ y) / (x @ x)            # Rayleigh quotient.
          res = cp.linalg.norm(y - lam * x)
          x = y / cp.linalg.norm(y)          # Normalize for next step.

        with nvtx.annotate(f"Copy {i}"):
          res_host = cp.asnumpy(res, blocking=False)
          x_host = cp.asnumpy(x, blocking=False)
          copy_event = cp.cuda.get_current_stream().record()

        with nvtx.annotate(f"Compute {i + 1} to {i + cfg.check_frequency}"):
          for j in range(i + 1, min(i + cfg.check_frequency, cfg.max_steps)):
            with nvtx.annotate(f"Compute Step {j}"):
              y = A_gpu @ x # We have to use `A_gpu` here as well.
              x = y / cp.linalg.norm(y) # Normalize for next step.

        with nvtx.annotate(f"I/O {i}", payload=i):
          copy_event.synchronize() # Wait for the copies to complete.

          if cfg.progress:
            print(f"step {i}: residual = {res_host:.3e}")

          # Save a checkpoint.
          np.savetxt(f"/tmp/device_{i}.txt", x_host)

          if res_host < cfg.residual_threshold:
            break

  return cp.asnumpy((x.T @ (A_gpu @ x)) / (x.T @ x)) # Copy from device to host.

Now let's make sure it works:

In [ ]:
lam_est_async = estimate_device_async(A_device)
np.testing.assert_allclose(lam_est_async, 1, atol=1e-4)

### 7. Performance Analysis

Before we profile the improved code, let's compare the execution times of both.

In [ ]:
quiet_cfg = PowerIterationConfig(progress=False)
power_iteration_baseline_duration = cpx.profiler.benchmark(
  estimate_device_baseline, (A_device, quiet_cfg), n_repeat=5, n_warmup=1
).cpu_times.mean() * 1000
power_iteration_async_duration = cpx.profiler.benchmark(
  estimate_device_async, (A_device, quiet_cfg), n_repeat=5, n_warmup=1
).cpu_times.mean() * 1000
speedup = power_iteration_baseline_duration / power_iteration_async_duration

print(f"power_iteration_baseline: {power_iteration_baseline_duration:.3f} ms")
print(f"power_iteration_async:    {power_iteration_async_duration:.3f} ms")
print(f"power_iteration_async speedup over power_iteration_baseline: {speedup:.2f}")

Next, let's capture a profile report of our improved code with the same Nsight Systems kernel.

In [ ]:
%%nsys -o power_iteration__async.nsys-rep
lam_est_async = estimate_device_async(A_device)
np.testing.assert_allclose(lam_est_async, 1, atol=1e-4)

Finally, let's look at the profile in Perfetto and confirm we've gotten rid of the idling.

In [ ]:
# The native report is saved as power_iteration__async.nsys-rep.

### 8. Balancing CPU I/O and GPU Compute

Our asynchronous implementation overlaps checkpoint I/O on the CPU with power-iteration steps on the GPU. `check_frequency` controls how many GPU steps run between residual checks and checkpoints. A smaller value produces output more frequently, but may not provide enough GPU work to hide the I/O. A larger value provides more GPU work to overlap with the I/O, but delays output and convergence checks.

**SOLUTION:** Sweep check frequencies from 20 through 35 in increments of 1 and determine the output frequency with the lowest execution time. Reuse the same matrix and benchmark `estimate_device_async` with progress disabled, so matrix setup remains outside the measurement.

In [ ]:
check_frequencies = list(range(20, 36))
async_durations = []

print("Sweeping async check frequencies...")
print("=" * 45)
print(f"{'Check Frequency':>18} | {'Time (ms)':>12}")
print("-" * 45)

for check_frequency in check_frequencies:
    cfg = PowerIterationConfig(check_frequency=check_frequency, progress=False)
    timing = cpx.profiler.benchmark(
        estimate_device_async, (A_device, cfg), n_repeat=5, n_warmup=1
    )
    duration_ms = timing.cpu_times.mean() * 1000
    async_durations.append(duration_ms)
    print(f"{check_frequency:>18} | {duration_ms:>9.3f} ms")

print("=" * 45)

In [ ]:
optimal_index = async_durations.index(min(async_durations))
optimal_check_frequency = check_frequencies[optimal_index]
optimal_duration = async_durations[optimal_index]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(check_frequencies, async_durations, 'b-o', linewidth=2, markersize=8)
ax.scatter(optimal_check_frequency, optimal_duration, color='red', s=100, zorder=3,
           label=f'Optimal: {optimal_check_frequency} steps')
for check_frequency, duration in zip(check_frequencies, async_durations):
    ax.text(check_frequency, duration, f'{duration:.1f}', va='bottom', ha='center')
ax.set_xticks(check_frequencies)
ax.set_xlabel('Check Frequency (steps)', fontsize=12)
ax.set_ylabel('Execution Time (ms)', fontsize=12)
ax.set_title('Async Power Iteration: Check Frequency Sweep', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Optimal check frequency: {optimal_check_frequency} steps ({optimal_duration:.3f} ms)')

Finally, profile the optimal check frequency and verify that the GPU compute between checks overlaps the CPU I/O.

In [ ]:
%%nsys -o power_iteration__async__optimal.nsys-rep
optimal_cfg = PowerIterationConfig(check_frequency=optimal_check_frequency)
lam_est_optimal = estimate_device_async(A_device, cfg=optimal_cfg)
np.testing.assert_allclose(lam_est_optimal, 1, atol=1e-4)